# Daily Challenge — Precision Prompting for Control & Output Quality

## Solution académique complète

Ce notebook suit un cycle complet de développement de prompt :

```text
Design → Generate → Evaluate → Detect issues → Refine → Re-evaluate
```

Le cas étudié est un extrait de politique interne sur la sécurité des accès
à distance.

Nous allons construire et tester des prompts qui contrôlent :

- le ton ;
- la structure ;
- le format ;
- la longueur ;
- la paraphrase ;
- l’extraction de citation ;
- la fidélité au texte source.

## Objectifs pédagogiques

À la fin du notebook, vous saurez :

1. construire un prompt avec des contraintes précises ;
2. limiter une sortie à un nombre maximal de mots ;
3. imposer un format en puces ;
4. différencier paraphrase, résumé et citation directe ;
5. détecter des ajouts non soutenus ;
6. réviser un prompt pour réduire les hallucinations ;
7. adapter le niveau de langue à une audience junior ;
8. évaluer automatiquement plusieurs contraintes.

## Texte source

L’ensemble du notebook utilise uniquement ce paragraphe :

> Employees must ensure that all remote access to internal systems is
> established via the approved secure VPN. Under no circumstances should
> unsecured connections or personal devices lacking endpoint protection be
> used to access proprietary data or sensitive communications.

In [ ]:
# Store the policy in one canonical variable so every test uses
# the exact same source text.
SOURCE_TEXT = (
    "Employees must ensure that all remote access to internal systems is "
    "established via the approved secure VPN. Under no circumstances "
    "should unsecured connections or personal devices lacking endpoint "
    "protection be used to access proprietary data or sensitive "
    "communications."
)

print(SOURCE_TEXT)

# 1. Prompt controlling tone, format and length

## Design goals

The prompt must:

- use a friendly and clear tone ;
- organize information with bullet points ;
- paraphrase instead of quoting ;
- stay under 75 words ;
- preserve every critical security rule.

## 1.1 First prompt version

In [ ]:
prompt_v1 = f"""
Act as an internal learning and communications writer.

Rewrite the policy text below as a short employee microlearning snippet.

<policy>
{SOURCE_TEXT}
</policy>

Requirements:
- Use a friendly, clear, and professional tone.
- Organize the content into bullet points.
- Paraphrase the policy. Do not copy complete sentences from the source.
- Keep the entire output under 75 words.
- Preserve the key requirements about:
  1. using the approved secure VPN;
  2. avoiding unsecured connections;
  3. not using personal devices without endpoint protection;
  4. protecting proprietary data and sensitive communications.
- Return only the final employee-facing snippet.
""".strip()

print(prompt_v1)

## 1.2 Example generated output

In a real workflow, replace this example with the response returned by the
chosen generative model.

In [ ]:
example_output_v1 = """
- Always use the company-approved secure VPN when accessing internal systems remotely.
- Never connect through an unsecured network.
- Do not use a personal device unless it has proper endpoint protection.
- These steps help protect company data and sensitive communications.
""".strip()

print(example_output_v1)

# 2. Evaluate the output

We evaluate:

- relevance ;
- clarity ;
- structure ;
- tone ;
- length ;
- factual accuracy.

Some criteria can be checked automatically. Others still require human
judgment.

In [ ]:
import re
from typing import Dict, List


def count_words(text: str) -> int:
    """Count words while handling apostrophes and hyphenated terms."""
    return len(
        re.findall(
            r"\b[\w]+(?:['’\-][\w]+)*\b",
            text,
            flags=re.UNICODE,
        )
    )


def non_empty_lines(text: str) -> List[str]:
    """Return only non-empty output lines."""
    return [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]


def is_bullet_line(line: str) -> bool:
    """Check whether a line begins with a common bullet marker."""
    return bool(re.match(r"^[-*•]\s+\S+", line))


def normalize_text(text: str) -> str:
    """Lowercase and normalize whitespace for simple comparisons."""
    return re.sub(r"\s+", " ", text.lower()).strip()

## 2.1 Critical policy concepts

We define a small checklist of concepts that must remain in the rewritten
content.

This is not a full semantic evaluator, but it helps detect missing policy
elements.

In [ ]:
REQUIRED_CONCEPTS = {
    "approved_vpn": [
        "approved secure vpn",
        "company-approved secure vpn",
        "approved vpn",
    ],
    "no_unsecured_connections": [
        "unsecured connection",
        "unsecured network",
        "avoid unsecured",
        "never connect through an unsecured",
    ],
    "personal_device_endpoint_protection": [
        "personal device",
        "endpoint protection",
    ],
    "protect_sensitive_information": [
        "proprietary data",
        "company data",
        "sensitive communications",
        "sensitive information",
    ],
}


def concept_is_present(
    text: str,
    alternatives: List[str],
) -> bool:
    """Return True when at least one acceptable phrase is present."""
    normalized = normalize_text(text)
    return any(
        alternative in normalized
        for alternative in alternatives
    )


def evaluate_policy_output(text: str) -> Dict:
    """Evaluate objective and semi-objective output constraints."""
    lines = non_empty_lines(text)

    concept_checks = {
        concept_name: concept_is_present(
            text,
            alternatives,
        )
        for concept_name, alternatives in REQUIRED_CONCEPTS.items()
    }

    friendly_markers = [
        "always",
        "remember",
        "help",
        "protect",
        "keep",
        "please",
        "these steps",
    ]

    return {
        "word_count": count_words(text),
        "under_75_words": count_words(text) < 75,
        "all_lines_are_bullets": (
            bool(lines)
            and all(is_bullet_line(line) for line in lines)
        ),
        "number_of_bullets": len(lines),
        "required_concepts": concept_checks,
        "all_required_concepts_present": all(
            concept_checks.values()
        ),
        "friendly_tone_marker_found": any(
            marker in normalize_text(text)
            for marker in friendly_markers
        ),
    }


evaluation_v1 = evaluate_policy_output(example_output_v1)
evaluation_v1

## 2.2 Human evaluation grid

In [ ]:
evaluation_grid = {
    "Relevance": {
        "status": "Pass",
        "comment": (
            "The output stays focused on remote-access security."
        ),
    },
    "Clarity": {
        "status": "Pass",
        "comment": (
            "The language is shorter and easier for non-technical staff."
        ),
    },
    "Structure": {
        "status": "Pass",
        "comment": "The content is organized into clear bullet points.",
    },
    "Tone": {
        "status": "Pass",
        "comment": (
            "The wording is supportive and suitable for internal communication."
        ),
    },
    "Length": {
        "status": (
            "Pass"
            if evaluation_v1["under_75_words"]
            else "Fail"
        ),
        "comment": (
            f"The output contains {evaluation_v1['word_count']} words."
        ),
    },
    "Factual Accuracy": {
        "status": (
            "Pass"
            if evaluation_v1["all_required_concepts_present"]
            else "Review"
        ),
        "comment": (
            "All four essential policy ideas are retained."
            if evaluation_v1["all_required_concepts_present"]
            else "At least one required policy concept is missing."
        ),
    },
}

evaluation_grid

## 2.3 Interpretation

The first output meets the main requirements:

- it is below 75 words ;
- it uses bullets ;
- it keeps the VPN rule ;
- it keeps the prohibition on unsecured access ;
- it keeps the endpoint-protection requirement ;
- it does not introduce a new technology or process.

# 3. Hallucination detection and mitigation

A hallucination in this task would be any unsupported addition, such as:

- mandatory multi-factor authentication ;
- a specific antivirus brand ;
- a requirement to report incidents within 24 hours ;
- a ban on all personal devices ;
- disciplinary consequences ;
- instructions to use public Wi-Fi with a hotspot.

None of these details appears in the source.

In [ ]:
# Terms that would be suspicious because they are not supported by the
# provided policy paragraph.
UNSUPPORTED_TERMS = [
    "multi-factor authentication",
    "mfa",
    "two-factor authentication",
    "2fa",
    "antivirus",
    "firewall",
    "24 hours",
    "report the incident",
    "disciplinary action",
    "mobile hotspot",
    "password manager",
    "company-issued device only",
]


def detect_unsupported_terms(text: str) -> List[str]:
    """Find known unsupported additions in a generated output."""
    normalized = normalize_text(text)

    return [
        term
        for term in UNSUPPORTED_TERMS
        if term in normalized
    ]


hallucination_check_v1 = detect_unsupported_terms(
    example_output_v1
)

print("Unsupported terms found:", hallucination_check_v1)

## 3.1 Example of a flawed output

The following output introduces new rules not present in the policy.

In [ ]:
flawed_output = """
- Use the approved VPN and multi-factor authentication every time.
- Only company-issued devices may access internal systems.
- Report any suspicious connection to IT within 24 hours.
""".strip()

print(flawed_output)
print(
    "\nUnsupported additions:",
    detect_unsupported_terms(flawed_output),
)

## 3.2 Revised prompt with strict grounding

In [ ]:
prompt_v2_grounded = f"""
Act as an internal policy communications editor.

Rewrite only the information contained in the source text below.

<source_policy>
{SOURCE_TEXT}
</source_policy>

NON-NEGOTIABLE RULES:
1. Paraphrase the source in a friendly and clear tone.
2. Use 3 or 4 bullet points.
3. Keep the entire output under 75 words.
4. Preserve every explicit requirement in the source.
5. Do not add new technologies, procedures, penalties, exceptions,
   recommendations, examples, timelines, or security controls.
6. Do not mention MFA, passwords, antivirus, firewalls, reporting
   procedures, public Wi-Fi, IT support, or company-issued devices unless
   those details appear in the source.
7. Do not remove the rule about personal devices lacking endpoint
   protection.
8. Do not quote complete source sentences.

Before answering, silently verify:
- every bullet is supported by the source;
- all critical requirements are preserved;
- the output is below 75 words;
- no external security advice has been added.

Return only the final bullet points.
""".strip()

print(prompt_v2_grounded)

## Why the revised prompt is safer

It adds:

- explicit source grounding ;
- a closed list of forbidden additions ;
- preservation rules ;
- a verification step ;
- a fallback against external knowledge.

# 4. Paraphrasing for junior interns

The audience changes from general employees to junior interns.

The prompt therefore requires:

- plain language ;
- short phrases ;
- no corporate or legal jargon ;
- no more than four bullets ;
- a supportive tone.

In [ ]:
junior_intern_prompt = f"""
Act as a friendly onboarding trainer for junior interns.

Rewrite the policy below in very simple language.

<policy>
{SOURCE_TEXT}
</policy>

Requirements:
- Use no more than 4 bullet points.
- Use short phrases and simple vocabulary.
- Avoid corporate, legal, and technical jargon whenever a simpler phrase
  is possible.
- Explain "VPN" as the company's approved secure connection.
- Keep the tone supportive and informative, not threatening.
- Preserve these rules:
  1. remote access must use the approved secure VPN;
  2. unsecured connections must not be used;
  3. personal devices without endpoint protection must not be used;
  4. company data and sensitive messages must be protected.
- Do not add new rules or recommendations.
- Keep the complete output under 65 words.
- Return only the bullet points.
""".strip()

print(junior_intern_prompt)

## 4.1 Example junior-friendly output

In [ ]:
junior_output = """
- Use the company’s approved secure connection whenever you work remotely.
- Do not access internal systems through an unsecured connection.
- Avoid personal devices that do not have endpoint protection.
- These rules help keep company information and sensitive messages safe.
""".strip()

print(junior_output)
print("\nEvaluation:")
print(evaluate_policy_output(junior_output))

## 4.2 Paraphrase quality checks

A good paraphrase should:

- preserve the original meaning ;
- change the wording and sentence structure ;
- avoid copying long sequences ;
- avoid weakening words such as “must” or “under no circumstances”.

In [ ]:
def longest_shared_word_sequence(
    source: str,
    candidate: str,
) -> int:
    """Compute the longest contiguous shared word sequence.

    This simple dynamic-programming function helps detect extensive copying.
    It is not a complete plagiarism detector.
    """
    source_words = re.findall(r"\b\w+\b", source.lower())
    candidate_words = re.findall(r"\b\w+\b", candidate.lower())

    # Each row stores matches ending at the current word pair.
    previous_row = [0] * (len(candidate_words) + 1)
    longest = 0

    for source_word in source_words:
        current_row = [0]

        for index, candidate_word in enumerate(
            candidate_words,
            start=1,
        ):
            if source_word == candidate_word:
                match_length = previous_row[index - 1] + 1
            else:
                match_length = 0

            current_row.append(match_length)
            longest = max(longest, match_length)

        previous_row = current_row

    return longest


shared_sequence_length = longest_shared_word_sequence(
    SOURCE_TEXT,
    junior_output,
)

print(
    "Longest identical contiguous word sequence:",
    shared_sequence_length,
)

# 5. Quote extraction variant

Quoting is appropriate when the exact wording has legal, policy, compliance
or audit importance.

The task here is to extract one direct quote that captures the central rule.

In [ ]:
quote_extraction_prompt = f"""
Act as a policy communications editor.

From the source policy below, extract exactly one direct quote that best
captures the core remote-access security requirement.

<policy>
{SOURCE_TEXT}
</policy>

Rules:
- Copy the selected words exactly.
- Do not paraphrase, correct, shorten internally, or combine separate parts.
- Select one complete sentence only.
- Put the quote inside quotation marks.
- After the quote, add one sentence of no more than 20 words explaining why
  it is the core rule.
- Do not add external interpretation.

Output format:
Quote: "..."
Why it matters: ...
""".strip()

print(quote_extraction_prompt)

## 5.1 Expected quote

In [ ]:
expected_quote = (
    "Employees must ensure that all remote access to internal systems is "
    "established via the approved secure VPN."
)

quote_output = (
    f'Quote: "{expected_quote}"\n'
    "Why it matters: It states the mandatory method for secure remote "
    "access to company systems."
)

print(quote_output)

## 5.2 Verify that the quote is exact

In [ ]:
def extract_quoted_text(output: str) -> str:
    """Extract the first double-quoted passage from an output."""
    match = re.search(r'"([^"]+)"', output)

    if not match:
        raise ValueError("No direct quote was found.")

    return match.group(1)


def verify_direct_quote(
    source: str,
    output: str,
) -> Dict:
    """Check whether the quoted text appears exactly in the source."""
    extracted_quote = extract_quoted_text(output)

    return {
        "quote": extracted_quote,
        "appears_exactly_in_source": extracted_quote in source,
        "is_expected_core_sentence": (
            extracted_quote == expected_quote
        ),
    }


quote_verification = verify_direct_quote(
    SOURCE_TEXT,
    quote_output,
)

quote_verification

# When quoting is more appropriate

Direct quotation is preferable in:

- formal compliance notices ;
- audit documentation ;
- policy acknowledgements ;
- mandatory training where exact wording must be preserved ;
- communications quoting a newly approved rule ;
- legal or disciplinary documentation.

In these contexts, precision and traceability are more important than
conversational simplicity.

# When quoting may create risk

Quoting can be risky when:

- a sentence is removed from important context ;
- the quoted text is outdated ;
- only part of a condition or exception is selected ;
- formal wording is too technical for the audience ;
- quotation marks suggest official approval where none exists ;
- the source contains confidential or sensitive information ;
- the quote is altered but still presented as exact.

A quote should therefore be checked against the source and presented with
enough context.

# 6. Full prompt development cycle

In [ ]:
prompt_cycle = {
    "design": (
        "Define audience, tone, bullet format, paraphrase requirement, "
        "critical concepts, and word limit."
    ),
    "generate": (
        "Run the prompt with a generative model and save the raw output."
    ),
    "evaluate": (
        "Check relevance, clarity, bullet structure, tone, word count, "
        "and preservation of policy facts."
    ),
    "detect": (
        "Search for unsupported technologies, procedures, timelines, "
        "penalties, and recommendations."
    ),
    "refine": (
        "Add strict grounding, forbidden additions, preservation rules, "
        "and a final verification instruction."
    ),
    "re_evaluate": (
        "Run the same checks on the revised output and compare results."
    ),
}

for stage, description in prompt_cycle.items():
    print(f"{stage.upper()}: {description}")

# 7. Comparative report

In [ ]:
outputs_to_compare = {
    "Employee version": example_output_v1,
    "Junior intern version": junior_output,
    "Flawed version": flawed_output,
}

comparison_rows = []

for output_name, output_text in outputs_to_compare.items():
    metrics = evaluate_policy_output(output_text)

    comparison_rows.append(
        {
            "output": output_name,
            "word_count": metrics["word_count"],
            "under_75_words": metrics["under_75_words"],
            "bullet_format": metrics["all_lines_are_bullets"],
            "all_required_concepts": (
                metrics["all_required_concepts_present"]
            ),
            "unsupported_terms": ", ".join(
                detect_unsupported_terms(output_text)
            ) or "None",
        }
    )

import pandas as pd

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

# 8. Final evaluation checklist

Before approving an AI-generated policy snippet, confirm:

1. Does it use only the provided source?
2. Are all mandatory policy requirements preserved?
3. Has any prohibition been weakened?
4. Are the bullets easy to scan?
5. Is the tone appropriate for the audience?
6. Is the output within the word limit?
7. Does it avoid unsupported tools, timelines or penalties?
8. Is paraphrased text genuinely paraphrased?
9. Is direct quoted text copied exactly?
10. Has a human reviewer checked factual accuracy?

# Conclusion

This exercise demonstrates that precision prompting requires more than a
well-written initial instruction.

A reliable workflow includes:

```text
Prompt design
    → generation
    → objective checks
    → human review
    → hallucination detection
    → prompt refinement
    → re-evaluation
```

Key lessons:

- constraints should be measurable ;
- critical source details should be listed explicitly ;
- anti-hallucination rules should forbid unsupported additions ;
- audience adaptation must not change the policy meaning ;
- paraphrases and direct quotes require different validation methods ;
- human review remains necessary for internal policy communication.